# Human-in-the-loop in LangChain
A beginner-friendly, runnable tutorial.

Imagine an assistant drafts an email. You want to review the recipient and wording **before** it sends anything. Human-in-the-loop (HITL) gives the agent a place to pause, wait for your decision, and continue.

You will learn to:

1. Pause before a tool executes.
2. Inspect the proposed action.
3. Approve it, edit it, or reject it.
4. Resume the same conversation.

This notebook runs without an API key or network requests after installation. A scripted model proposes a fixed email, and a simulated tool records it in a Python list. No real email is sent.

## 1. The idea
An ordinary agent can propose a tool call and execute it immediately. With HITL middleware, a configured tool call waits for review:

```text
User asks for an action
        |
Model proposes a tool call
        |
Middleware pauses BEFORE the tool runs
        |
Human reviews the name and arguments
        |
        +-- Approve --> Execute the original call
        +-- Edit ----> Execute with corrected arguments
        +-- Reject --> Skip the call and give feedback
        |
Agent continues
```

The pause is enforced by application code. Asking the model to "please get approval" in a prompt alone does not implement this pause.

In this tutorial, **you act as the reviewer by inspecting one cell's output and running the next cell with your decision**. The examples show each possible decision separately.

## 2. Setup
Use Python 3.11+ and the project's `.venv` Jupyter kernel. If imports fail, uncomment the installation line, run it, and restart the kernel.

The examples were verified with LangChain 1.4.3, langchain-core 1.6.3, and LangGraph 1.2.11. LangChain supplies the agent and approval middleware; its underlying LangGraph runtime supplies saved state and the resume command.

In [ ]:
# Uncomment if the packages are missing in your notebook environment.
# %pip install "langchain>=1.4.3,<2" "langchain-core>=1.6.3,<2" "langgraph>=1.2.11,<2"

from copy import deepcopy
from pprint import pprint

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langchain_core.language_models.fake_chat_models import FakeMessagesListChatModel
from langchain_core.messages import AIMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

## 3. A tool whose execution we can see
Our `send_email` tool only appends to `outbox`. An empty outbox means the tool has not executed. This makes the effect of approval visible without setting up an email service.

Avoid calling this tool directly in the review flow: the middleware only controls calls made through the agent.

In [ ]:
outbox = []

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Simulate sending an email by recording it in the notebook's outbox."""
    email = {"to": to, "subject": subject, "body": body}
    outbox.append(email)
    return f"Simulated email recorded for {to}."

print("Outbox:", outbox)

## 4. A predictable model for learning
A real model decides which tool to call and what arguments to use. To keep this tutorial focused on approval, we use LangChain's fake chat model with two scripted replies:

1. Propose a `send_email` call.
2. Return a final acknowledgement after review is processed.

The small adapter below makes that fake model accept tools. It does not perform language understanding, validate a tool choice, or generate an email from the user's text. The **agent, middleware, pause, tool execution, and resume are real LangChain behavior**.

You can skim this helper on your first read; the HITL configuration is in the next section.

In [ ]:
DRAFT = {
    "to": "alex@example.com",
    "subject": "Meeting reminder",
    "body": "Hi Alex, our meeting is at 3 PM.",
}

class ScriptedToolModel(FakeMessagesListChatModel):
    def bind_tools(self, tools, **kwargs):
        # The scripted response already contains the tool call.
        return self

def make_demo_model():
    return ScriptedToolModel(
        responses=[
            AIMessage(
                content="",
                tool_calls=[{
                    "name": "send_email",
                    "args": deepcopy(DRAFT),
                    "id": "email-call",
                    "type": "tool_call",
                }],
            ),
            AIMessage(content="Review processed. See the tool result for the outcome."),
        ]
    )

## 5. Configure the approval gate
Three pieces work together:

| Piece | Purpose |
| --- | --- |
| `HumanInTheLoopMiddleware` | Pauses before the configured tool runs |
| `InMemorySaver` | Keeps the paused agent state in this Python process |
| `thread_id` | Identifies the conversation to pause and resume |

Here, every `send_email` call requires review. We explicitly allow `approve`, `edit`, and `reject`.

We create a fresh agent and thread for each example so the three decisions can be explored independently. The in-memory saver is enough for this notebook; it loses its saved state when the kernel restarts.

In [ ]:
def make_review_agent():
    return create_agent(
        model=make_demo_model(),
        tools=[send_email],
        middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "send_email": {
                        "allowed_decisions": ["approve", "edit", "reject"],
                    }
                },
                description_prefix="Please review this email before sending",
            )
        ],
        checkpointer=InMemorySaver(),
    )

def new_config():
    # A new ID lets you rerun an example without reusing an old conversation.
    from uuid import uuid4
    return {
        "configurable": {"thread_id": str(uuid4())},
        "recursion_limit": 10,
    }

REQUEST = {
    "messages": [{
        "role": "user",
        "content": "Send Alex a reminder that our meeting is at 3 PM.",
    }]
}

## 6. Run until the agent pauses
Run the next cell, then stop and inspect its output.

The default `invoke` result contains `__interrupt__` when review is required. Each interrupt's `value` contains the proposed actions and their review policies. At this point, the email tool has **not** run.

The current documentation also shows a `version="v2"` result format with an `.interrupts` attribute. This notebook uses the default dictionary result consistently.

In [ ]:
outbox.clear()
approve_agent = make_review_agent()
approve_config = new_config()

paused = approve_agent.invoke(REQUEST, config=approve_config)

assert paused.get("__interrupt__"), "Expected an approval pause."
review = paused["__interrupt__"][0].value
pprint(review)
print("\nOutbox BEFORE approval:", outbox)
assert outbox == [], "The tool must not execute before review."

## 7. Approve the proposed email
After inspecting the draft, run this cell to approve it.

`Command(resume=...)` delivers your decision to the waiting agent. Use the **same agent and config**, especially the same `thread_id`. Supplying the original user request again would start another turn rather than deliver a review decision.

Approving runs the tool with its original arguments. Expect one email in the outbox.

In [ ]:
approved = approve_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    config=approve_config,
)

print("Outbox AFTER approval:")
pprint(outbox)
for message in approved["messages"]:
    if isinstance(message, ToolMessage):
        print("Tool result:", message.content)

assert not approved.get("__interrupt__")
assert outbox == [DRAFT]
approved_outbox = deepcopy(outbox)

## 8. Edit before execution
Now start a separate example. The model proposes the same draft, but you notice that the meeting should be at **4 PM**.

Run the first cell to inspect the pending action. Then run the second cell to deliver the edited action. An `edit` decision both changes the call and allows it to execute; it does not create a second review pause in this example.

We copy the proposed `args` shown in the interrupt. The resume payload uses `edited_action` with `name` and `args`.

In [ ]:
outbox.clear()
edit_agent = make_review_agent()
edit_config = new_config()

edit_paused = edit_agent.invoke(REQUEST, config=edit_config)
assert edit_paused.get("__interrupt__")
edit_action = edit_paused["__interrupt__"][0].value["action_requests"][0]
pprint(edit_action)
print("Outbox BEFORE editing:", outbox)
assert outbox == []

In [ ]:
edited_args = deepcopy(edit_action["args"])
edited_args["body"] = "Hi Alex, our meeting is at 4 PM."

edited = edit_agent.invoke(
    Command(resume={
        "decisions": [{
            "type": "edit",
            "edited_action": {
                "name": edit_action["name"],
                "args": edited_args,
            },
        }]
    }),
    config=edit_config,
)

print("Outbox AFTER editing:")
pprint(outbox)
assert not edited.get("__interrupt__")
assert outbox == [edited_args]
assert "4 PM" in outbox[0]["body"]
edited_outbox = deepcopy(outbox)

## 9. Reject the action
In a third example, imagine the meeting is cancelled.

Run the first cell to inspect the proposal. Then run the second cell to reject it. The tool should be skipped and the outbox should remain empty. The rejection `message` becomes feedback in a tool-result message, so the model can decide how to continue.

Our scripted model simply acknowledges the outcome. A real model may propose another action after rejection; any new call to a gated tool must be reviewed again.

In [ ]:
outbox.clear()
reject_agent = make_review_agent()
reject_config = new_config()

reject_paused = reject_agent.invoke(REQUEST, config=reject_config)
assert reject_paused.get("__interrupt__")
pprint(reject_paused["__interrupt__"][0].value["action_requests"])
assert outbox == []

In [ ]:
rejected = reject_agent.invoke(
    Command(resume={
        "decisions": [{
            "type": "reject",
            "message": "The meeting is cancelled. Do not send or retry this email.",
        }]
    }),
    config=reject_config,
)

print("Outbox AFTER rejection:", outbox)
for message in rejected["messages"]:
    if isinstance(message, ToolMessage):
        print("Feedback to model:", message.content)

assert not rejected.get("__interrupt__")
assert outbox == []
rejected_outbox = deepcopy(outbox)

## 10. Compare the outcomes
The same proposed tool call produced three different outcomes because of the reviewer's decision:

| Decision | Does the tool run? | Arguments |
| --- | --- | --- |
| Approve | Yes | Original draft |
| Edit | Yes | Corrected draft |
| Reject | No | Feedback is returned instead |

The checks below verify the effects recorded by the tool, rather than relying on the model's final wording.

In [ ]:
assert approved_outbox[0]["body"] == "Hi Alex, our meeting is at 3 PM."
assert edited_outbox[0]["body"] == "Hi Alex, our meeting is at 4 PM."
assert rejected_outbox == []
print("Verified: approval executes, editing changes the arguments, rejection skips execution.")

## 11. Try it yourself
1. Change the edited recipient to `sam@example.com`, then rerun the edit example from its pause cell.
2. Change the rejection feedback and inspect the tool-result message.
3. Temporarily set `"send_email": False` in the factory and run a new agent. It should execute without review; the assertion expecting a pause will fail. Restore the policy afterwards.

To use a real model later, replace `make_demo_model()` with a tool-calling chat model configured for your provider. Keep the middleware, checkpointer, thread config, and resume payloads. The reviewer must inspect the actual generated action, since it will no longer be fixed.

### Common mistakes
- **Missing checkpointer:** the agent needs saved state to pause and resume.
- **Different thread ID on resume:** the decision must go back to the waiting conversation.
- **Running the resume cell twice:** a decision applies to a pending interrupt. Restart the example from its pause cell to repeat it.
- **Only printing a draft:** displaying an action is not the same as enforcing a pause before execution.
- **Assuming rejection stops the entire agent:** it denies the reviewed tool call; the agent can continue.
- **Kernel restart:** rerun the notebook; `InMemorySaver` does not persist across process restarts.

For a real application, your UI or application code collects the human decision and constructs the same resume command. If several actions are paused together, supply one decision per action in the order shown. A persistent checkpointer is needed to resume after a service restart.

### References
- [LangChain human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) — approval configuration and decision payloads.
- [LangChain agents](https://docs.langchain.com/oss/python/langchain/agents) — creating agents and invoking them.
- [LangGraph persistence](https://docs.langchain.com/oss/python/langgraph/persistence) — checkpointers and conversation threads.